In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("TripExpansion") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# --- Loading scoped West Yorkshire timetable patterns ---
timetables = spark.read.csv("../data/interim/timetables_west_yorkshire.csv", header=True, inferSchema=True)
timetables = timetables.repartition(4).cache()
print("Pattern rows:", timetables.count())
print("Partitions:", timetables.rdd.getNumPartitions())

26/07/14 10:21:07 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Pattern rows: 34703
Partitions: 4


In [2]:
# --- Building the calendar table: one row per date, April-June 2026 ---
calendar = spark.sql("""
    SELECT explode(sequence(to_date('2026-04-01'), to_date('2026-06-30'), interval 1 day)) AS trip_date
""")
calendar = calendar.withColumn("weekday_name", F.date_format("trip_date", "EEEE"))
print("Calendar rows:", calendar.count())
calendar.show(5)

Calendar rows: 91
+----------+------------+
| trip_date|weekday_name|
+----------+------------+
|2026-04-01|   Wednesday|
|2026-04-02|    Thursday|
|2026-04-03|      Friday|
|2026-04-04|    Saturday|
|2026-04-05|      Sunday|
+----------+------------+
only showing top 5 rows


In [3]:
# --- Explode days_of_week into individual weekday rows ---
timetables_exploded = timetables.withColumn(
    "weekday_name", F.explode(F.split(F.col("days_of_week"), ","))
)

# --- Broadcast join: small calendar table joined against exploded patterns ---
from pyspark.sql.functions import broadcast

trip_instances = timetables_exploded.join(
    broadcast(calendar), on="weekday_name", how="inner"
)

trip_instances = trip_instances.cache()
print("Total expanded trip instances:", trip_instances.count())
print("Partitions after join:", trip_instances.rdd.getNumPartitions())

[Stage 16:>                                                         (0 + 4) / 4]

Total expanded trip instances: 1186094
Partitions after join: 4


In [4]:
trip_instances.select(
    "operator_name", "line_name", "origin", "destination",
    "trip_date", "weekday_name", "departure_time", "service_code"
).show(10)

+--------------------+---------+--------+-----------+----------+------------+-------------------+------------+
|       operator_name|line_name|  origin|destination| trip_date|weekday_name|     departure_time|service_code|
+--------------------+---------+--------+-----------+----------+------------+-------------------+------------+
|The Keighley Bus ...|       66|Keighley|    Skipton|2026-06-27|    Saturday|2026-07-14 13:20:00| PB0001748:6|
|The Keighley Bus ...|       66|Keighley|    Skipton|2026-06-20|    Saturday|2026-07-14 13:20:00| PB0001748:6|
|The Keighley Bus ...|       66|Keighley|    Skipton|2026-06-13|    Saturday|2026-07-14 13:20:00| PB0001748:6|
|The Keighley Bus ...|       66|Keighley|    Skipton|2026-06-06|    Saturday|2026-07-14 13:20:00| PB0001748:6|
|The Keighley Bus ...|       66|Keighley|    Skipton|2026-05-30|    Saturday|2026-07-14 13:20:00| PB0001748:6|
|The Keighley Bus ...|       66|Keighley|    Skipton|2026-05-23|    Saturday|2026-07-14 13:20:00| PB0001748:6|
|

**Error: Got today's date when this Spark Job ran instead of actual departure_time**

**Fix: read it explicitly as a string, then combine with trip_date**

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
from pyspark.sql.types import StringType

timetables = spark.read.csv("../data/interim/timetables_west_yorkshire.csv", header=True, inferSchema=False)
timetables = timetables.repartition(4).cache()
print("Pattern rows:", timetables.count())
timetables.printSchema()

Pattern rows: 34703
root
 |-- source_file: string (nullable = true)
 |-- operator_name: string (nullable = true)
 |-- national_operator_code: string (nullable = true)
 |-- service_code: string (nullable = true)
 |-- line_name: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- days_of_week: string (nullable = true)
 |-- operating_start_date: string (nullable = true)
 |-- journey_code: string (nullable = true)
 |-- departure_time: string (nullable = true)



**Rebuilding the calendar:**

In [6]:
calendar = spark.sql("""
    SELECT explode(sequence(to_date('2026-04-01'), to_date('2026-06-30'), interval 1 day)) AS trip_date
""")
calendar = calendar.withColumn("weekday_name", F.date_format("trip_date", "EEEE"))
print("Calendar rows:", calendar.count())

Calendar rows: 91


Correctly building the real date time

In [7]:
timetables_exploded = timetables.withColumn(
    "weekday_name", F.explode(F.split(F.col("days_of_week"), ","))
)

trip_instances = timetables_exploded.join(
    broadcast(calendar), on="weekday_name", how="inner"
)

trip_instances = trip_instances.withColumn(
    "scheduled_datetime",
    F.to_timestamp(F.concat_ws(" ", F.col("trip_date").cast("string"), F.col("departure_time")))
)

trip_instances = trip_instances.cache()
print("Total expanded trip instances:", trip_instances.count())

[Stage 38:>                                                         (0 + 4) / 4]

Total expanded trip instances: 1186094


In [8]:
trip_instances.select(
    "operator_name", "line_name", "trip_date", "departure_time", "scheduled_datetime"
).show(10, truncate=False)

+-------------+---------+----------+--------------+-------------------+
|operator_name|line_name|trip_date |departure_time|scheduled_datetime |
+-------------+---------+----------+--------------+-------------------+
|Team Pennine |356      |2026-06-29|18:28:00      |2026-06-29 18:28:00|
|Team Pennine |356      |2026-06-22|18:28:00      |2026-06-22 18:28:00|
|Team Pennine |356      |2026-06-15|18:28:00      |2026-06-15 18:28:00|
|Team Pennine |356      |2026-06-08|18:28:00      |2026-06-08 18:28:00|
|Team Pennine |356      |2026-06-01|18:28:00      |2026-06-01 18:28:00|
|Team Pennine |356      |2026-05-25|18:28:00      |2026-05-25 18:28:00|
|Team Pennine |356      |2026-05-18|18:28:00      |2026-05-18 18:28:00|
|Team Pennine |356      |2026-05-11|18:28:00      |2026-05-11 18:28:00|
|Team Pennine |356      |2026-05-04|18:28:00      |2026-05-04 18:28:00|
|Team Pennine |356      |2026-04-27|18:28:00      |2026-04-27 18:28:00|
+-------------+---------+----------+--------------+-------------

In [9]:
# dedupe: the same scheduled trip (same operator, line, date, time, service) can appear
# more than once if the same route was published across multiple XML files/operators —
# keep one row per unique trip before saving
key_cols = ["operator_name", "line_name", "trip_date", "departure_time", "service_code"]

before_count = trip_instances.count()
trip_instances = trip_instances.dropDuplicates(key_cols)
after_count = trip_instances.count()

print(f"Rows before dedup: {before_count}")
print(f"Rows after dedup: {after_count}")
print(f"Duplicates removed: {before_count - after_count}")

[Stage 52:>                                                         (0 + 4) / 4]

Rows before dedup: 1186094
Rows after dedup: 765089
Duplicates removed: 421005


In [10]:
trip_instances.write.mode("overwrite").parquet("../data/interim/trip_instances_expanded")
print("Saved.")

26/07/14 10:21:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Saved.


In [11]:
trip_instances.count()

765089

In [12]:
print(spark.sparkContext.uiWebUrl)

http://localhost:4043
